# Домашнее задание 4: PWM and Markov Models

In [8]:
!pip install biopython

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from Bio import SeqIO, motifs
from Bio.Seq import Seq
import warnings
warnings.filterwarnings('ignore')

BACKGROUND = {'A': 0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}
NUCLEOTIDES = ['A', 'C', 'G', 'T']
BG_PROBS = np.array([BACKGROUND[n] for n in NUCLEOTIDES])

---
# Часть I: Моделирование мотивов и PWM

## Задание 1. Реализация PWM «с нуля»

In [10]:
sites = [
    "GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA",
    "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC",
    "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC",
    "CAGGTTGGC"
]
L = len(sites[0])
N = len(sites)

**PFM** — считаем, сколько раз каждый нуклеотид встречается на каждой позиции.

In [11]:
def compute_pfm(sites):
    L = len(sites[0])
    pfm = np.zeros((4, L), dtype=float)
    nuc_idx = {n: i for i, n in enumerate(NUCLEOTIDES)}
    for seq in sites:
        for pos, nuc in enumerate(seq):
            pfm[nuc_idx[nuc], pos] += 1
    return pfm

pfm = compute_pfm(sites)
print('PFM (строки: A,C,G,T | столбцы: позиции 1..9):')
print(pfm)

PFM (строки: A,C,G,T | столбцы: позиции 1..9):
[[ 1.  8.  1.  0.  0.  2.  7.  2.  1.]
 [ 6.  2.  1.  0.  0.  6.  0.  0.  8.]
 [ 1.  0.  8. 10.  0.  0.  3.  8.  0.]
 [ 2.  0.  0.  0. 10.  2.  0.  0.  1.]]


**PPM** — нормализуем PFM в вероятности, добавляя псевдосчёт α=0.1, чтобы избежать нулевых вероятностей:

$$PPM_{b,i} = \frac{count_{b,i} + \alpha}{N + 4\alpha}$$

In [12]:
def pfm_to_ppm(pfm, alpha=0.1):
    N = pfm.sum(axis=0)
    return (pfm + alpha) / (N + 4 * alpha)

ppm = pfm_to_ppm(pfm)
np.set_printoptions(precision=4)
print('PPM:')
print(ppm)
print('Суммы по столбцам:', ppm.sum(axis=0).round(6))

PPM:
[[0.1058 0.7788 0.1058 0.0096 0.0096 0.2019 0.6827 0.2019 0.1058]
 [0.5865 0.2019 0.1058 0.0096 0.0096 0.5865 0.0096 0.0096 0.7788]
 [0.1058 0.0096 0.7788 0.9712 0.0096 0.0096 0.2981 0.7788 0.0096]
 [0.2019 0.0096 0.0096 0.0096 0.9712 0.2019 0.0096 0.0096 0.1058]]
Суммы по столбцам: [1. 1. 1. 1. 1. 1. 1. 1. 1.]


**PWM** — логарифм отношения вероятности к фону. Положительное значение означает, что нуклеотид встречается чаще случайного, отрицательное — реже:

$$PWM_{b,i} = \log_2\frac{PPM_{b,i}}{background_b}$$

In [13]:
def ppm_to_pwm(ppm, background=BACKGROUND):
    bg = np.array([background[n] for n in NUCLEOTIDES]).reshape(-1, 1)
    return np.log2(ppm / bg)

pwm = ppm_to_pwm(ppm)
print('PWM (log-odds):')
print(pwm)

PWM (log-odds):
[[-1.4798  1.4006 -1.4798 -4.9392 -4.9392 -0.5469  1.2105 -0.5469 -1.4798]
 [ 1.5166 -0.0218 -0.9547 -4.4141 -4.4141  1.5166 -4.4141 -4.4141  1.9257]
 [-0.9547 -4.4141  1.9257  2.2441 -4.4141 -4.4141  0.5401  1.9257 -4.4141]
 [-0.5469 -4.9392 -4.9392 -4.9392  1.719  -0.5469 -4.9392 -4.9392 -1.4798]]


Максимальный скор достигается, если на каждой позиции выбрать нуклеотид с наибольшим весом, минимальный — с наименьшим.

In [14]:
def score_sequence(seq, pwm):
    nuc_idx = {n: i for i, n in enumerate(NUCLEOTIDES)}
    return sum(pwm[nuc_idx[nuc], pos] for pos, nuc in enumerate(seq))

max_seq = ''.join(NUCLEOTIDES[i] for i in pwm.argmax(axis=0))
min_seq = ''.join(NUCLEOTIDES[i] for i in pwm.argmin(axis=0))

print(f'Максимальный скор: {pwm.max(axis=0).sum():.4f}   {max_seq}')
print(f'Минимальный скор:  {pwm.min(axis=0).sum():.4f}    {min_seq}')

Максимальный скор: 15.3846   CAGGTCAGC
Минимальный скор:  -39.9434    ATTAAGTTG
